In [1]:
from pathlib import Path
import pandas as pd

df = pd.read_csv("superstore.csv")

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (10194, 23)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Returned,Regional Manager
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,No,Roxanne Rodriguez
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,No,Roxanne Rodriguez
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717,No,Roxanne Rodriguez
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,No,Roxanne Rodriguez
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,No,Chuck Magee


In [2]:
# Column Descriptions

column_descriptions = {
    "Row ID": "Unique identifier assigned to each row in the dataset.",
    "Order ID": "Unique identifier for each customer order.",
    "Order Date": "Date when the order was placed.",
    "Ship Date": "Date when the order was shipped.",
    "Ship Mode": "Shipping method selected for the order.",
    "Customer ID": "Unique identifier for each customer.",
    "Customer Name": "Name of the customer who placed the order.",
    "Segment": "Customer segment, such as Consumer, Corporate, or Home Office.",
    "Country/Region": "Country or region where the order was placed.",
    "City": "City associated with the order.",
    "State": "State associated with the order.",
    "Postal Code": "Postal code of the delivery location.",
    "Region": "Business region associated with the order.",
    "Product ID": "Unique identifier for each product.",
    "Category": "Main product category, such as Technology or Furniture.",
    "Sub-Category": "More specific classification of a product.",
    "Product Name": "Name or description of the product.",
    "Sales": "Revenue generated from the product sale.",
    "Quantity": "Number of product units ordered.",
    "Discount": "Discount applied to the product sale.",
    "Profit": "Profit or loss generated from the sale.",
    "Returned": "Indicates whether the order was returned.",
    "Regional Manager": "Manager responsible for the corresponding region."
}

description_df = pd.DataFrame([
    {
        "Column": column,
        "Description": description
    }
    for column, description in column_descriptions.items()
    if column in df.columns
])

display(description_df)

description_df.to_csv(
    "outputs/column_descriptions.csv",
    index=False
)

print("Column descriptions saved to outputs/column_descriptions.csv")

,Column,Description
0,Row ID,Unique identifier assigned to each row in the ...
1,Order ID,Unique identifier for each customer order.
2,Order Date,Date when the order was placed.
3,Ship Date,Date when the order was shipped.
4,Ship Mode,Shipping method selected for the order.
5,Customer ID,Unique identifier for each customer.
6,Customer Name,Name of the customer who placed the order.
7,Segment,"Customer segment, such as Consumer, Corporate,..."
8,Country/Region,Country or region where the order was placed.
9,City,City associated with the order.


Column descriptions saved to outputs/column_descriptions.csv


## 2. Validate the columns used in the analysis

In [3]:
required_columns = [
    "Sales", "Profit", "Order ID", "Customer ID",
    "Category", "Sub-Category", "Product Name",
    "Product ID", "Segment", "Region"
]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("Missing values in KPI columns:")
display(df[["Sales", "Profit", "Order ID"]].isnull().sum())

print("\nData types:")
display(df[["Sales", "Profit", "Order ID"]].dtypes)

print("\nUnique categories:", sorted(df["Category"].dropna().unique()))
print("Unique products:", df["Product ID"].nunique())

Missing values in KPI columns:


Sales       0
Profit      0
Order ID    0
dtype: int64


Data types:


Sales       float64
Profit      float64
Order ID        str
dtype: object


Unique categories: ['Furniture', 'Office Supplies', 'Technology']
Unique products: 1862


## 3. Calculate the four core KPIs

In [4]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = (total_profit / total_sales * 100) if total_sales else 0
total_orders = df["Order ID"].nunique()

print("========== SUPERSTORE KPI REPORT ==========")
print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Profit Margin: {profit_margin:.2f}%")
print(f"Total Orders: {total_orders:,}")

========== SUPERSTORE KPI REPORT ==========
Total Sales: $2,326,534.35
Total Profit: $292,296.81
Profit Margin: 12.56%
Total Orders: 5,111


## 4. Category performance

In [5]:
category_analysis = (
    df.groupby("Category")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum")
      )
      .reset_index()
      .sort_values("Total_Sales", ascending=False)
)

display(category_analysis.round(2))

,Category,Total_Sales,Total_Profit
2,Technology,839893.28,146543.38
0,Furniture,754747.76,19730.00
1,Office Supplies,731893.31,126023.44


## 5. Sub-category performance

In [6]:
subcategory_analysis = (
    df.groupby("Sub-Category")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum")
      )
      .reset_index()
      .sort_values("Total_Sales", ascending=False)
)

display(subcategory_analysis.round(2))

,Sub-Category,Total_Sales,Total_Profit
5,Chairs,335768.25,27223.53
13,Phones,331842.64,45050.83
14,Storage,224644.55,21285.11
16,Tables,208020.18,-17753.21
3,Binders,207354.88,31426.10
11,Machines,189925.03,3461.98
0,Accessories,167380.32,41936.64
6,Copiers,150745.29,56093.94
4,Bookcases,115361.20,-3632.07
1,Appliances,108213.18,18329.48


## 6. Customer segment performance

In [7]:
segment_analysis = (
    df.groupby("Segment")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Orders=("Order ID", "nunique")
      )
      .reset_index()
      .sort_values("Total_Sales", ascending=False)
)

display(segment_analysis.round(2))

,Segment,Total_Sales,Total_Profit,Total_Orders
0,Consumer,1170659.79,136371.45,2628
1,Corporate,715806.13,94249.64,1552
2,Home Office,440068.43,61675.73,931


## 7. Regional performance

In [8]:
region_analysis = (
    df.groupby("Region")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Orders=("Order ID", "nunique")
      )
      .reset_index()
      .sort_values("Total_Sales", ascending=False)
)

display(region_analysis.round(2))

,Region,Total_Sales,Total_Profit,Total_Orders
3,West,739813.61,110798.82,1635
1,East,691828.17,94883.26,1475
0,Central,503170.67,39865.31,1179
2,South,391721.90,46749.43,822


## 8. Save the results

In [9]:
output_folder = Path("outputs")
output_folder.mkdir(parents=True, exist_ok=True)

kpi_summary = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Profit Margin (%)", "Total Orders"],
    "Value": [total_sales, total_profit, profit_margin, total_orders]
})

kpi_summary.to_csv(output_folder / "kpi_summary.csv", index=False)
category_analysis.to_csv(output_folder / "category_analysis.csv", index=False)
subcategory_analysis.to_csv(output_folder / "subcategory_analysis.csv", index=False)
segment_analysis.to_csv(output_folder / "segment_analysis.csv", index=False)
region_analysis.to_csv(output_folder / "region_analysis.csv", index=False)

print("Saved these files in:", output_folder.resolve())
print("- kpi_summary.csv")
print("- category_analysis.csv")
print("- subcategory_analysis.csv")
print("- segment_analysis.csv")
print("- region_analysis.csv")

Saved these files in: C:\Users\Ahraf\Downloads\project_analysis\new\outputs
- kpi_summary.csv
- category_analysis.csv
- subcategory_analysis.csv
- segment_analysis.csv
- region_analysis.csv
